We now have a minimal implementation of CAD operations. It's time to test our new API against a provider.

We'll port over the Blender implementation and test running it with the new API.

In [ ]:
from codetocad.adapters.blender import *

In [ ]:
!uv pip install -e "..[blender]"

## Installing CodeToCAD inside Blender

Call the code cell below at least once to install CodeToCAD. 

After that, you can run scripts inside Blender using `run_blender()`.

In [ ]:
from codetocad.adapters.blender import (
    install_codetocad_in_blender,
    set_blender_executable_path,
)

set_blender_executable_path("/Applications/Blender.app/Contents/MacOS/Blender")

install_codetocad_in_blender()

## Running scripts in Blender

The new API allows you to run code inside Blender straight from this Jupyter notebook or a python script. 

No need to install an addon anymore, simplifying UX and allowing you to focus on just modeling!

Below is a Cube in the Blende API, it's quite verbose, and it's not even a cube. The codetocad implementaiton is a few cells down.

In [ ]:
def create_cube():
    """Sketching and creating a cube in Blender"""
    import bpy
    from codetocad.adapters.blender.blender_actions.context import (
        get_context_view_3d,
    )  # cheating to get ahead
    from codetocad.adapters.blender.blender_actions.context import update_view_layer

    # Clear scene
    bpy.ops.object.select_all(action="SELECT")
    bpy.ops.object.delete()

    # Define square sketch points (2D)
    points = [(1, 1, 0), (1, -1, 0), (-1, -1, 0), (-1, 1, 0), (1, 1, 0)]

    # Create a new curve
    curve_data = bpy.data.curves.new(name="SquareSketch", type="CURVE")
    curve_data.dimensions = "2D"
    polyline = curve_data.splines.new("POLY")
    polyline.points.add(len(points) - 1)

    for i, coord in enumerate(points):
        x, y, z = coord
        polyline.points[i].co = (x, y, z, 1)

    # Make object
    curve_obj = bpy.data.objects.new("SquareSketch", curve_data)
    bpy.context.collection.objects.link(curve_obj)

    bpy.ops.object.select_all(action="SELECT")
    # Convert curve to mesh
    with get_context_view_3d():
        # bpy.context.view_layer.objects.active = existing_object
        update_view_layer()

        bpy.context.view_layer.objects.active = curve_obj
        bpy.ops.object.convert(target="MESH")

        # Enter edit mode, select all, extrude along Z
        bpy.ops.object.mode_set(mode="EDIT")
        bpy.ops.mesh.select_all(action="SELECT")
        bpy.ops.mesh.extrude_region_move(TRANSFORM_OT_translate={"value": (0, 0, 2)})
        bpy.ops.object.mode_set(mode="OBJECT")

        cube = bpy.context.active_object
        cube.name = "SketchCube"

In [ ]:
from codetocad.adapters.blender import run_blender

run_blender(create_cube, background=False)

### Visualize using Open3D

In [ ]:
from platform import system

import open3d as o3d

from codetocad.cli.config import get_temp_stl_export_path

if system() != "Darwin":
    from open3d.web_visualizer import draw
else:
    # The web visualizer is not available on macOS.
    from open3d.visualization import draw

## Taking the new API for a spin

It's time to test the new API in Blender.

In [ ]:
def create_cube():
    """Create a simple cube using the new Blender CAD implementation."""

    from codetocad.cli.config import get_temp_stl_export_path

    try:
        print("🎯 Creating a cube using Blender CAD implementation...")

        # Create a cube part using the preset
        cube = Part.preset.cube(2, 2, 2)
        cube.set_name("test_cube")

        print(f"✅ Successfully created cube: {cube}")
        print(f"   Cube name: {cube.name}")
        print(f"   Blender object: {cube.get_blender_object()}")

        assembly = Assembly()
        assembly.add(cube)
        assembly.export.stl(str(get_temp_stl_export_path()))

    except Exception as e:
        print(f"❌ Error creating cube: {e}")
        import traceback

        traceback.print_exc()


run_blender(create_cube, background=True)


mesh = o3d.io.read_triangle_mesh(str(get_temp_stl_export_path()))
mesh.paint_uniform_color([0.5, 0.5, 0.5])  # Gray color

draw(mesh)

In [ ]:
def test_vertex_edge():
    """Test basic vertex and edge creation."""
    try:
        print("🔧 Testing vertex and edge creation...")

        # Create vertices
        v1 = Vertex(0, 0, 0, name="origin")
        v2 = Vertex(1, 1, 1, name="corner")

        print(f"   Created vertex 1: {v1}")
        print(f"   Created vertex 2: {v2}")

        # Create edge
        edge = Edge(v1, v2, name="test_edge")
        print(f"   Created edge: {edge}")
        print(f"   Edge length: {edge.geometry.length()}")

        return edge

    except Exception as e:
        print(f"❌ Error in vertex/edge test: {e}")
        import traceback

        traceback.print_exc()
        return None


run_blender(test_vertex_edge, background=False)

In [ ]:
def test_assembly():
    """Test assembly creation with multiple parts."""
    try:
        print("🏗️ Testing assembly creation...")

        # Create an assembly
        assembly = Assembly("test_assembly")

        # Create parts
        cube = Part.preset.cube(1, 1, 1)
        cube.set_name("assembly_cube")

        cylinder = Part.preset.cylinder(0.5, 2)
        cylinder.set_name("assembly_cylinder")

        # Add parts to assembly
        assembly.add(cube)
        assembly.add(cylinder)

        print(f"   Created assembly: {assembly}")
        print(f"   Assembly parts: {len(assembly.parts)}")

        return assembly

    except Exception as e:
        print(f"❌ Error in assembly test: {e}")
        import traceback

        traceback.print_exc()
        return None


run_blender(test_assembly, background=False)